# **Cabezas que copian**: ¿hay cabezas que mueven el loop pero no la copia legítima? — Qwen3.5-0.8B (Colab L4)

El mapa de daño mostró que el parche de loops `p50c` es un "anti-repetición": siempre prendido, daña nombres que se
repiten del contexto, el segundo "step by step", líneas repetidas de un doctest. Aquí buscamos cabezas de atención
que muevan **solo** la copia dentro del loop.

Sobre respuestas del base guardadas en Drive (sin generar), cada respuesta es una ventana de hasta 4,096 tokens
alrededor de donde se dispara el detector de texto. Tres tipos de token: **dentro del loop**, **copia legítima**
(fuera del loop, predecible copiando el contexto) y **resto**.

1. Mitad A: se apaga cada una de las 336 cabezas (6 capas de atención × 8 + 18 capas DeltaNet × 16) y se mide cuánto
   sube la NLL en cada tipo de token.
2. Se ordenan por "sube en loop − sube en copia legítima"; las mejores 1/2/4/8 se apagan juntas en la **mitad B**
   (apartada), contra cabezas al azar y contra el parche `p50c` siempre prendido.
3. Correlación entre "pega al loop" y "pega a la copia legítima" en todas las cabezas: alta = el loop usa la misma
   maquinaria de copia (no hay atajo por cabezas); baja = hay cabezas propias del loop.

~35–45 min en L4. Resultado: `MyDrive/wsi_runs/FINALES/copy_heads_qwen35-08b.json`. Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "copy_heads_qwen35-08b"
OWN_RUN = "discover_blind"     # respuestas del base guardadas en MyDrive/wsi_runs (GSM8K)
REPO = "/content/wsi"
BASE = "/content/base"
S = f"{REPO}/scripts/experiment_copy_heads.py"
PATCH = f"{REPO}/patches/qwen35_0.8b_p50c_loop_fix.moipatch"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"   # causal-conv1d compiles from source for ages: skip it
import torch, transformers; print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, shutil, json, subprocess, datetime, time
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/wsi_runs"
FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
os.makedirs(FINALES, exist_ok=True); os.makedirs(OUTD, exist_ok=True)
W = "/content/work_gsm8k"
src = f"{ROOT}/{OWN_RUN}"
need = ("config.json", "requests_test.jsonl", "outputs_test_base.jsonl")
assert all(os.path.exists(f"{src}/{f}") for f in need), f"faltan respuestas guardadas en {src}"
os.makedirs(W, exist_ok=True)
for f in need:
    shutil.copy2(f"{src}/{f}", W)
print("respuestas <-", src)

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert os.path.exists(S), "el repo no tiene experiment_copy_heads.py: falta push"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
out = f"{OUTD}/heads.json"
if not os.path.exists(out):                       # retoma: lo ya hecho no se repite
    subprocess.run(["python", "-u", S, "--work", W, "--model", BASE, "--patch", PATCH, "--out", "/content/heads.json"],
                   check=True, cwd=f"{REPO}/scripts")
    shutil.copy2("/content/heads.json", out)
RES = json.load(open(out))

In [ ]:
print(f"tokens mitad B: {RES['tokens_b']}")
print(f"correlación (pega al loop vs pega a la copia legítima) entre cabezas: {RES['spearman_loop_vs_copy']}")
print("\nmejores cabezas (mitad A): sube NLL en loop / copia / resto")
for s in RES["top_by_spec"][:8]:
    print(f"  {s['module']}#{s['head']:<3d} {s['loop']:+.3f} / {s['copy']:+.3f} / {s['other']:+.3f}")
print("\nmitad B (apartada): sube NLL en loop / copia / resto")
for k, v in RES["combos_b"].items():
    t, r = v["ablate_top"], v["ablate_random"]
    print(f"  mejores {k}: {t['loop']:+.3f} / {t['copy']:+.3f} / {t['other']:+.3f}   | al azar {k}: {r['loop']:+.3f} / {r['copy']:+.3f} / {r['other']:+.3f}")
if "patch_b" in RES:
    p = RES["patch_b"]
    print(f"  parche p50c siempre prendido: {p['loop']:+.3f} / {p['copy']:+.3f} / {p['other']:+.3f}")
final = {"run": RES}
final["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "copy_heads_colab.ipynb",
                   "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                   "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(final, fh, indent=1)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")